In [ ]:
import Pkg; Pkg.status();

In [ ]:
using Revise
using Makie
using SolidStateDetectors
const CSG = SolidStateDetectors.ConstructiveSolidGeometry
using Unitful
T = Float32
import CairoMakie
using GeometryBasics
using StatsBase
using LaTeXStrings
using Geant4
using Colors
using Random
using ArraysOfArrays, LinearAlgebra
using RadiationDetectorDSP
using ProgressMeter 

const colors = RGB{Float64}[RGB(0.0, 0.6056031704619725, 0.9786801190138923), RGB(0.8888735440600661, 0.435649148506399, 0.2781230452972766), RGB(0.24222393333911896, 0.6432750821113586, 0.304448664188385), RGB(0.7644400000572205, 0.4441118538379669, 0.8242975473403931), RGB(0.6755439043045044, 0.5556622743606567, 0.09423444420099258), RGB(0.0, 0.6657590270042419, 0.6809969544410706), RGB(0.9307674765586853, 0.3674771189689636, 0.5757699012756348), RGB(0.776981770992279, 0.5097429752349854, 0.14642538130283356), RGB(5.29969987894674e-8, 0.6642677187919617, 0.5529508590698242), RGB(0.558464765548706, 0.59348464012146, 0.11748137325048447), RGB(0.0, 0.6608786582946777, 0.7981787919998169), RGB(0.609670877456665, 0.49918484687805176, 0.9117812514305115), RGB(0.38000133633613586, 0.5510532855987549, 0.9665056467056274), RGB(0.9421815872192383, 0.3751642107963562, 0.4518167972564697), RGB(0.8684020638465881, 0.39598923921585083, 0.7135148048400879), RGB(0.4231467843055725, 0.6224954128265381, 0.19877080619335175)];

In [ ]:
using FileIO
logo = FileIO.load(joinpath(dirname(pathof(SolidStateDetectors)), "../docs/src/assets/logo.png"));

In [ ]:
sim = Simulation{T}("GERDA50A.yaml");

In [ ]:
function Makie.convert_arguments(::Type{<:Makie.Mesh}, surf::CSG.AbstractSurfacePrimitive)
    m = CSG.mesh(surf, 40)
    faces = NgonFace{length(first(m.connections))}.(m.connections)
    (GeometryBasics.mesh(Point3.(m.x, m.y, m.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), )
end

In [ ]:
Makie.@recipe(DetectorPlot, det) do scene
    Makie.Attributes(
        interpolate = true,
        transparency = true,
        shading = Makie.NoShading,
        alpha = 0.2
    )
end
Makie.preferred_axis_type(plot::DetectorPlot) = Makie.Axis3

In [ ]:
function Makie.plot!(p::DetectorPlot{<:Tuple{<:SolidStateDetector}})
    
    det = p.det[]

    for (i,c) in enumerate(det.contacts)
        for surf in CSG.surfaces(c.geometry)
            Makie.mesh!(p, p.attributes, surf; color = colors[i])
        end
    end

    if !ismissing(det.passives)
        for (i,c) in enumerate(det.passives)
            for surf in CSG.surfaces(c.geometry)
                Makie.mesh!(p, p.attributes, surf; color = :gray)
            end
        end
    end
    
    p
end

In [ ]:
length_unit = u"mm"
fig = Makie.Figure(size = (500,500))
ax = Makie.Axis3(fig[1,1], 
    aspect = :data, 
    dim1_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim2_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim3_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    xlabel = "x / $length_unit",
    ylabel = "y / $length_unit",
    zlabel = "z / $length_unit",
    xspinecolor_2 = :transparent,
    xspinecolor_3 = :transparent,
    xspinecolor_4 = :transparent,
    yspinecolor_2 = :transparent,
    yspinecolor_3 = :transparent,
    yspinecolor_4 = :transparent,
    zspinecolor_2 = :transparent,
    zspinecolor_3 = :transparent,
    zspinecolor_4 = :transparent,
    azimuth = π/6 - π/2,
    elevation = π/6,
    clip_decorations = false
)

detectorplot!(ax, sim.detector)
fig

# Source with cone emission

In [ ]:
source = MonoenergeticSource("gamma", 2.615u"MeV", CartesianPoint(0.15,0,0.04), CartesianVector(-1,0,0), 45u"°")
app = G4JLApplication(sim, source);

In [ ]:
_evts = run_geant4_simulation(app, 5_000)

In [ ]:
# plot the distribution
let m = source, l = 0.01
    if iszero(m.opening_angle)
        v = m.position
        vnew = m.position + length * normalize(m.direction)
        Makie.lines!([v.x, vnew.x], [v.y, vnew.y], [v.z, vnew.z], color = :green, linewidth = 2)
    elseif m.opening_angle <= 90u"°"
        d = normalize(m.direction)
        a = normalize(d × (abs(d.x) == 1 ? CartesianVector(0,1,0) : CartesianVector(1,0,0)))
        b = normalize(a × d)
        rot = hcat(a,b,d)
        cone = SolidStateDetectors.ConstructiveSolidGeometry.Cone(r = ((0,0),(0,l*sin(m.opening_angle))), hZ = l*cos(m.opening_angle)/2, 
        origin = m.position + rot * [0,0,l*cos(m.opening_angle)/2], 
        rotation = rot)
        
        for surf in CSG.surfaces(cone)
            ms = CSG.mesh(surf, 40)
            faces = if length(first(ms.connections)) == 3
                vcat(ms.connections'...)
            else
                vcat(
                    getindex.(ms.connections, Ref(1:3))'...,
                    getindex.(ms.connections, Ref([3,4,1]))'...
                )
            end |> Makie.to_triangles
            Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(ms.x, ms.y, ms.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
                interpolate = true, transparency = true, color = :green, shading = Makie.NoShading)
        end
    end
    
    Makie.scatter!([m.position.x * u"m"], [m.position.y * u"m"], [m.position.z * u"m"], markersize = 15, color = :gray, strokecolor = :black, strokewidth = 1, label = string(typeof(m).name.name))
end

pt = first.(_evts.pos)
Makie.scatter!(ax, getindex.(pt, 1) .* u"m", getindex.(pt, 2) .* u"m", getindex.(pt, 3) .* u"m", color = :black, markersize = 0.5)
fig

In [ ]:
Random.seed!(123)
# evts = run_geant4_simulation(app, 250_000)
evts = run_geant4_simulation(app, 1_000_000)
E = add_fano_noise.(sum.(evts.edep), 2.95u"eV", 0.129);

In [ ]:
using LegendHDF5IO
lh5open("GERDA50A_G4_events_z_40mm.lh5", "w") do h
    LegendHDF5IO.writedata(h, "sim", evts)
end

In [ ]:
using LegendHDF5IO
evts = lh5open("GERDA50A_G4_events_z_40mm.lh5", "r") do h; LegendHDF5IO.readdata(h, "sim") end
Random.seed!(123)
E = add_fano_noise.(sum.(evts.edep), 2.95u"eV", 0.129);

In [ ]:
h = StatsBase.fit(Histogram, ustrip.(u"keV", E), 0:10:3000)
Makie.stephist(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = :green,
    axis = (xgridvisible = false, ygridvisible = false, 
        limits = (0,3000,500,5e5), yscale = Makie.log10,
        xlabel = "Energy / keV", ylabel = "Counts / 10 keV"
    ))
Makie.hist!(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges),
    weights = replace(h.weights, 0 => 1e-10), 
    color = (:green, 0.3))
Makie.text!(2615, 2.5e5, text = L"$^{208}$Tl FEP", align = (:center, :bottom))
Makie.text!(2615 - 511, 3.2e4, text = L"$^{208}$Tl SEP", align = (:center, :bottom))
Makie.text!(2615 - 2*511, 1.3e4, text = L"$^{208}$Tl DEP", align = (:center, :bottom))
Makie.current_figure()

In [ ]:
length_unit = u"mm"
scale = 0.7
fig = Makie.Figure(size = (1300,500) .* scale, background_color = :transparent)
gs = Makie.GridLayout(fig[1,1])
ax = Makie.Axis3(gs[1,1], 
    aspect = :data, 
    limits = (-51,51,-51,51,-11,91),
    xticks = Makie.WilkinsonTicks(5),
    yticks = Makie.WilkinsonTicks(5),
    dim1_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim2_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim3_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    xlabel = "x / $length_unit",
    ylabel = "y / $length_unit",
    zlabel = "z / $length_unit         ",
    xspinecolor_2 = :transparent,
    xspinecolor_3 = :transparent,
    xspinecolor_4 = :transparent,
    yspinecolor_2 = :transparent,
    yspinecolor_3 = :transparent,
    yspinecolor_4 = :transparent,
    zspinecolor_2 = :transparent,
    zspinecolor_3 = :transparent,
    zspinecolor_4 = :transparent,
    azimuth = π/6 - π/2,
    elevation = π/6,
    clip_decorations = false
)

for (i,c) in enumerate(sim.detector.contacts)
    surfs = CSG.surfaces(c.geometry)
    for surf in surfs 
        m = CSG.mesh(surf, 40)
        faces = if length(first(m.connections)) == 3
            vcat(m.connections'...)
        else
            vcat(
                getindex.(m.connections, Ref(1:3))'...,
                getindex.(m.connections, Ref([3,4,1]))'...
            )
        end |> Makie.to_triangles
        Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(m.x, m.y, m.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
            rasterize = 4, interpolate = true, transparency = true, alpha = 0.2, color = colors[i], shading = Makie.NoShading)
    end
end


# plot the source emission
let m = source, l = 0.01
    if iszero(m.opening_angle)
        v = m.position
        vnew = m.position + length * normalize(m.direction)
        Makie.lines!([v.x, vnew.x], [v.y, vnew.y], [v.z, vnew.z], color = :green, linewidth = 2)
    elseif m.opening_angle <= 90u"°"
        d = normalize(m.direction)
        a = normalize(d × (abs(d.x) == 1 ? CartesianVector(0,1,0) : CartesianVector(1,0,0)))
        b = normalize(a × d)
        rot = hcat(a,b,d)
        cone = SolidStateDetectors.ConstructiveSolidGeometry.Cone(r = ((0,0),(0,l*sin(m.opening_angle))), hZ = l*cos(m.opening_angle)/2, 
        origin = m.position + rot * [0,0,l*cos(m.opening_angle)/2], 
        rotation = rot)
        
        for surf in CSG.surfaces(cone)
            ms = CSG.mesh(surf, 40)
            faces = if length(first(ms.connections)) == 3
                vcat(ms.connections'...)
            else
                vcat(
                    getindex.(ms.connections, Ref(1:3))'...,
                    getindex.(ms.connections, Ref([3,4,1]))'...
                )
            end |> Makie.to_triangles
            Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(ms.x, ms.y, ms.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
                rasterize = 4, interpolate = true, transparency = true, color = :green, shading = Makie.NoShading)
        end
    end
    
    Makie.scatter!(ax, [m.position.x * u"m"], [m.position.y * u"m"], [m.position.z * u"m"], 
        rasterize = 4, markersize = 10, color = :gray, strokecolor = :black, strokewidth = 1, 
        label = string(typeof(m).name.name))
end

# plot the distribution of hits in the detector
pt = first.(evts.pos)[1:min(1_000_000, end)]
Makie.scatter!(ax, getindex.(pt, 1) .* u"m", getindex.(pt, 2) .* u"m", getindex.(pt, 3) .* u"m", 
    color = :black, markersize = 0.2, rasterize = 4)

ax2 = Makie.Axis(gs[1,2], aspect = Makie.DataAspect())
Makie.poly!(Point2f[(-1,-0.5), (1,-0.5), (1,-1), (2,0), (1,1), (1,0.5), (-1,0.5)], color = :green, strokecolor = :black, strokewidth = 1)
Makie.hidedecorations!(ax2)
Makie.hidespines!(ax2)

# Right plot
ax3 = Makie.Axis(gs[1,3], xgridvisible = false, ygridvisible = false, 
        limits = (0,3000,5e2,5e5), xticks = 0:500:3000, yscale = Makie.log10,
        xlabel = "Energy / keV", ylabel = "Counts / 10 keV")



h = StatsBase.fit(Histogram, ustrip.(u"keV", E), 0:10:3000)
Makie.stephist!(ax3,
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = :green)
Makie.hist!(ax3,
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = (:green, 0.3))
Makie.text!(ax3, 2615, 2.5e5, text = "²⁰⁸Tl FEP", align = (:center, :bottom))
Makie.text!(ax3, 2615 - 511, 3.2e4, text = "²⁰⁸Tl SEP", align = (:center, :bottom))
Makie.text!(ax3, 2615 - 2*511, 1.3e4, text = "²⁰⁸Tl DEP", align = (:center, :bottom))

Makie.colsize!(gs, 2, Makie.Auto(0.2))
Makie.colsize!(gs, 3, Makie.Auto(1.5))
Makie.colgap!(gs, 30 * scale)

fig

# Makie.save("Event_distribution_Geant4.pdf", fig)

# 228Th flood measurement

In [ ]:
## MAKE SURE THE CRYOSTAT IS PLACED CORRECTLY TO SHIELD THE ELECTRONS

In [ ]:
# sim = Simulation{T}(SSD_examples[:InvertedCoaxInCryostat])
source = IsotopeSource(90, 228, 0.0, 0.0, CartesianPoint(0.15,0,0.05))#, CartesianVector(-1,0,0), 2u"°")

In [ ]:
#source = IsotopeSource(90, 228, 0.0, 0.0, CartesianPoint(0.02,0,0.0801), CartesianVector(0,0,-1), 0u"°")
#source = IsotopeSource(81, 208, 0.0, 0.0, CartesianPoint(0.06,0,0005), CartesianVector(-1,0,0), 10u"°")
#source = MonoenergeticSource("gamma", 2.615u"MeV", CartesianPoint(0.05,0,0.05), CartesianVector(-1,0,0), 180u"°")
app = G4JLApplication(sim, source);

In [ ]:
using LegendHDF5IO
seed = 1234 # 1234
Random.seed!(seed)
evts = run_geant4_simulation(app, 250_000)#, energy_threshold = 1.5u"MeV")

In [ ]:
lh5open("GERDA50A_events_isotropic_seed_$(seed).lh5", "cw") do h
    LegendHDF5IO.writedata(h, "data", evts)
end

In [ ]:
using LegendHDF5IO
using TypedTables

_evts = vcat((lh5open("GERDA50A_events_isotropic_seed_$(seed).lh5", "r") do h
    LegendHDF5IO.readdata(h, "data")
end for seed in (1234,))...)

Random.seed!(1234)
E = add_fano_noise.(sum.(_evts.edep), 2.95u"eV", 0.129)

# _evts = lh5open("G4_events_seed_1234.lh5", "r") do h
#     LegendHDF5IO.readdata(h, "data")
# end

# separate pileup into separate events

elem_ptr = deepcopy(_evts.thit.elem_ptr)
kernel_size = deepcopy(_evts.thit.kernel_size)
added = 0
for i in eachindex(_evts.thit)
    thit = _evts.thit[i]
    d = findall(diff(thit) .> zero(eltype(thit)))
    isempty(d) && continue
    splice!(elem_ptr, i + added, elem_ptr[i + added] .+ [0; d])
    append!(kernel_size, (() for _ in d))
    added += length(d)
end

evts = Table(merge(NamedTuple(c => begin
    if c == :evtno 
        collect(eachindex(kernel_size))
    else
        f = getproperty(_evts, c)
        VectorOfArrays(f.data, elem_ptr, kernel_size)
    end end
for c in columnnames(_evts))))

# evts = SolidStateDetectors.cluster_detector_hits(evts, 0.05u"mm")

In [ ]:
length_unit = u"mm"
fig = Makie.Figure(size = (500,500), backgroundcolor = :transparent)
ax = Makie.Axis3(fig[1,1], 
    aspect = :data, 
    dim1_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim2_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim3_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    xlabel = "x / $length_unit",
    ylabel = "y / $length_unit",
    zlabel = "z / $length_unit",
    xspinecolor_2 = :transparent,
    xspinecolor_3 = :transparent,
    xspinecolor_4 = :transparent,
    yspinecolor_2 = :transparent,
    yspinecolor_3 = :transparent,
    yspinecolor_4 = :transparent,
    zspinecolor_2 = :transparent,
    zspinecolor_3 = :transparent,
    zspinecolor_4 = :transparent,
    azimuth = π/6 - π/2,
    elevation = π/6,
    clip_decorations = false
)

detectorplot!(ax, sim.detector)
# for (i,c) in enumerate(sim.detector.contacts)
#     surfs = CSG.surfaces(c.geometry)
#     for surf in surfs 
#         m = CSG.mesh(surf, 40)
#         faces = if length(first(m.connections)) == 3
#             vcat(m.connections'...)
#         else
#             vcat(
#                 getindex.(m.connections, Ref(1:3))'...,
#                 getindex.(m.connections, Ref([3,4,1]))'...
#             )
#         end |> Makie.to_triangles
#         Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(m.x, m.y, m.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
#             interpolate = true, transparency = true, alpha = 0.2, color = colors[i], shading = Makie.NoShading)
#     end
# end
fig

In [ ]:
# plot the distribution
let m = source, l = 0.005, evts = view(evts, 1:min(length(evts), 5_000))
    if iszero(m.opening_angle)
        v = m.position .* u"m"
        vnew = (m.position .+ l .* normalize(m.direction)) .* u"m"
        Makie.lines!([v.x, vnew.x], [v.y, vnew.y], [v.z, vnew.z], color = :green, linewidth = 2)
    elseif m.opening_angle <= 90u"°"
        d = normalize(m.direction)
        a = normalize(d × (abs(d.x) == 1 ? CartesianVector(0,1,0) : CartesianVector(1,0,0)))
        b = normalize(a × d)
        rot = hcat(a,b,d)
        cone = SolidStateDetectors.ConstructiveSolidGeometry.Cone(r = ((0,0),(0,l*sin(m.opening_angle))), hZ = l*cos(m.opening_angle)/2, 
        origin = m.position + rot * [0,0,l*cos(m.opening_angle)/2], 
        rotation = rot)
        
        for surf in CSG.surfaces(cone)
            ms = CSG.mesh(surf, 40)
            faces = if length(first(ms.connections)) == 3
                vcat(ms.connections'...)
            else
                vcat(
                    getindex.(ms.connections, Ref(1:3))'...,
                    getindex.(ms.connections, Ref([3,4,1]))'...
                )
            end |> Makie.to_triangles
            Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(ms.x, ms.y, ms.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
                interpolate = true, transparency = true, color = :green, shading = Makie.NoShading)
        end
    end
    
    Makie.scatter!([m.position.x * u"m"], [m.position.y * u"m"], [m.position.z * u"m"], markersize = 15, color = :gray, strokecolor = :black, strokewidth = 1, label = string(typeof(m).name.name))
end

pt = first.(evts.pos)
Makie.scatter!(ax, getindex.(pt, 1) .* u"m", getindex.(pt, 2) .* u"m", getindex.(pt, 3) .* u"m", color = :black, markersize = 0.5)
fig

In [ ]:
length_unit = u"mm"
scale = 0.7
fig = Makie.Figure(size = (1300,500) .* scale, backgroundcolor = :transparent)
gs = Makie.GridLayout(fig[1,1])
ax = Makie.Axis3(gs[1,1], 
    aspect = :data, 
    limits = (-51,51,-51,51,-11,91),
    xticks = Makie.WilkinsonTicks(5),
    yticks = Makie.WilkinsonTicks(5),
    dim1_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim2_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    dim3_conversion = Makie.UnitfulConversion(length_unit, units_in_label = false),
    xlabel = "x / $length_unit",
    ylabel = "y / $length_unit",
    zlabel = "z / $length_unit         ",
    xspinecolor_2 = :transparent,
    xspinecolor_3 = :transparent,
    xspinecolor_4 = :transparent,
    yspinecolor_2 = :transparent,
    yspinecolor_3 = :transparent,
    yspinecolor_4 = :transparent,
    zspinecolor_2 = :transparent,
    zspinecolor_3 = :transparent,
    zspinecolor_4 = :transparent,
    azimuth = π/6 - π/2,
    elevation = π/6,
    clip_decorations = false,
    backgroundcolor = :white
)

# add contacts
for (i,c) in enumerate(sim.detector.contacts)
    surfs = CSG.surfaces(c.geometry)
    for surf in surfs 
        m = CSG.mesh(surf, 40)
        faces = if length(first(m.connections)) == 3
            vcat(m.connections'...)
        else
            vcat(
                getindex.(m.connections, Ref(1:3))'...,
                getindex.(m.connections, Ref([3,4,1]))'...
            )
        end |> Makie.to_triangles
        Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(m.x, m.y, m.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
            rasterize = 4, interpolate = true, transparency = true, alpha = 0.2, color = colors[i], shading = Makie.NoShading)
    end
end

# add cryostat
for (i,c) in enumerate(sim.detector.passives)
    surfs = CSG.surfaces(c.geometry)
    for surf in surfs 
        m = CSG.mesh(surf, 40)
        faces = if length(first(m.connections)) == 3
            vcat(m.connections'...)
        else
            vcat(
                getindex.(m.connections, Ref(1:3))'...,
                getindex.(m.connections, Ref([3,4,1]))'...
            )
        end |> Makie.to_triangles
        Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(m.x, m.y, m.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
            rasterize = 4, interpolate = true, transparency = true, alpha = 0.2, color = (:black, 0.25), shading = Makie.NoShading)
    end
end


# plot the source emission
let m = source, l = 0.01
    if iszero(m.opening_angle)
        v = m.position
        vnew = m.position + length * normalize(m.direction)
        Makie.lines!([v.x, vnew.x], [v.y, vnew.y], [v.z, vnew.z], color = :green, linewidth = 2)
    elseif m.opening_angle <= 90u"°"
        d = normalize(m.direction)
        a = normalize(d × (abs(d.x) == 1 ? CartesianVector(0,1,0) : CartesianVector(1,0,0)))
        b = normalize(a × d)
        rot = hcat(a,b,d)
        cone = SolidStateDetectors.ConstructiveSolidGeometry.Cone(r = ((0,0),(0,l*sin(m.opening_angle))), hZ = l*cos(m.opening_angle)/2, 
        origin = m.position + rot * [0,0,l*cos(m.opening_angle)/2], 
        rotation = rot)
        
        for surf in CSG.surfaces(cone)
            ms = CSG.mesh(surf, 40)
            faces = if length(first(ms.connections)) == 3
                vcat(ms.connections'...)
            else
                vcat(
                    getindex.(ms.connections, Ref(1:3))'...,
                    getindex.(ms.connections, Ref([3,4,1]))'...
                )
            end |> Makie.to_triangles
            Makie.mesh!(ax, GeometryBasics.Mesh(Point3.(ms.x, ms.y, ms.z) .* uconvert(NoUnits, 1.0 * SolidStateDetectors.internal_length_unit/length_unit), faces), 
                rasterize = 4, interpolate = true, transparency = true, color = :green, shading = Makie.NoShading)
        end
    end
    
    Makie.scatter!(ax, [m.position.x * u"m"], [m.position.y * u"m"], [m.position.z * u"m"], 
        rasterize = 4, markersize = 10, color = :gray, strokecolor = :black, strokewidth = 1, 
        label = string(typeof(m).name.name))
end

# plot the distribution of hits in the detector
pt = flatview(_evts.pos)[1:min(1_000_000, end)]
Makie.scatter!(ax, getindex.(pt, 1) .* u"m", getindex.(pt, 2) .* u"m", getindex.(pt, 3) .* u"m", 
    color = :black, markersize = 0.2, rasterize = 4)

ax2 = Makie.Axis(gs[1,2], aspect = Makie.DataAspect())
Makie.poly!(Point2f[(-1,-0.5), (1,-0.5), (1,-1), (2,0), (1,1), (1,0.5), (-1,0.5)], color = :green, strokecolor = :black, strokewidth = 1)
Makie.hidedecorations!(ax2)
Makie.hidespines!(ax2)

# Right plot
ax3 = Makie.Axis(gs[1,3], xgridvisible = false, ygridvisible = false, 
        limits = (0,2800,50,50e3), xticks = 0:500:3000, yscale = Makie.log10,
        xlabel = "Energy / keV", ylabel = "Counts / 10 keV")

h = StatsBase.fit(Histogram, ustrip.(u"keV", E), 0:10:2630)
h.weights[end] = 0
Makie.stephist!(ax3,
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = :green)
Makie.hist!(ax3,
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    fillto = 1e-10,
    color = (:green, 0.3))
Makie.text!(ax3, 2615 + 150, 5e3, text = "²⁰⁸Tl FEP", align = (:right, :bottom))
Makie.text!(ax3, 2615 - 511, 8e2, text = "²⁰⁸Tl SEP", align = (:center, :bottom))
Makie.text!(ax3, 2615 - 2*511, 5.2e2, text = "²⁰⁸Tl DEP", align = (:center, :bottom))

Makie.colsize!(gs, 2, Makie.Auto(0.2))
Makie.colsize!(gs, 3, Makie.Auto(1.5))
Makie.colgap!(gs, 30 * scale)

FileIO.save("event_distribution_spectrum.pdf", fig)
fig

# Compose pulse shape parameters

In [ ]:
using LegendHDF5IO
evts = lh5open("GERDA50A_G4_events_z_40mm.lh5", "r") do h; LegendHDF5IO.readdata(h, "sim") end
Random.seed!(123)
E = add_fano_noise.(sum.(evts.edep), 2.95u"eV", 0.129);

In [ ]:
h = StatsBase.fit(Histogram, ustrip.(u"keV", E), 1500:0.25:1650)
Makie.stephist(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = :green,
    axis = (xgridvisible = false, ygridvisible = false, 
        limits = (1500,1650,0,nothing), #yscale = Makie.log10,
        xlabel = "Energy / keV", ylabel = "Counts / 0.25 keV"
    ))
Makie.hist!(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges),
    weights = replace(h.weights, 0 => 1e-10), 
    color = (:green, 0.3))
Makie.vlines!([1591,1595])
Makie.current_figure()

In [ ]:
sim.detector = SolidStateDetector(sim.detector, ADL2016ChargeDriftModel{T}(temperature = 95u"K"))
sim.detector = SolidStateDetector(sim.detector, ConstantImpurityDensity{T}(-5e9u"cm^-3"))
sim.detector = SolidStateDetector(sim.detector, BoggsChargeTrappingModel{T}(Dict("nσe-1" => 0.5e4u"cm", "nσh-1" => 1e4u"cm")))

In [ ]:
calculate_weighting_potential!(sim, 1, refinement_limits = [0.2,0.1,0.05,0.02,0.01])

In [ ]:
calculate_electric_potential!(sim, refinement_limits = [0.2,0.1,0.05,0.02,0.01])

In [ ]:
calculate_electric_field!(sim)

In [ ]:
DEP_evts = SolidStateDetectors.cluster_detector_hits(evts[findall(1591u"keV" .< E .< 1595u"keV")], 0.1u"mm")
DEP_pos = SolidStateDetectors.flatview(SolidStateDetectors.cluster_detector_hits(DEP_evts, 0.5u"mm").pos)
Makie.scatter(hypot.(getindex.(DEP_pos,1), getindex.(DEP_pos,2)), getindex.(DEP_pos,3), axis = (aspect = DataAspect(), ), markersize = 1, color = :black)

In [ ]:
SolidStateDetectors.radius_guess(::T, ::Type{SolidStateDetectors.Gamma}) where {T} = T(2e-5)

In [ ]:
sideband_evts = SolidStateDetectors.cluster_detector_hits(evts[findall(1585u"keV" .< E .< 1610u"keV")], 0.1u"mm")
SolidStateDetectors.simulate_waveforms(sideband_evts, sim, ".", "GERDA50A_95K_simulated_waveforms_", number_of_carriers = 40, number_of_shells = 1, diffusion = true, self_repulsion = true, Δt = 1u"ns", max_nsteps = 3_000, n_threads = 8);

In [ ]:
using LegendHDF5IO

wf = vcat(
    (lh5open("GERDA50A_95K_simulated_waveforms_evts_$(lpad(i*1000+1,5,'0'))-$(lpad((i+1)*1000,5,'0')).h5") do h
    LegendHDF5IO.readdata(h, "generated_waveforms")
        end for i in 0:14)...
)

In [ ]:
Random.seed!(123)
E = add_fano_noise.(sum.(wf.edep), 2.95u"eV", 0.12);
Ectc = add_fano_noise.(last.(getfield.(wf.waveform, :signal)) * 2.95u"V", 2.95u"eV", 0.12);

In [ ]:
h = StatsBase.fit(Histogram, ustrip.(u"keV", E), 0:0.2:3000)
fig, ax, _ = Makie.stephist(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges), 
    weights = replace(h.weights, 0 => 1e-10), 
    color = :green,
    label = "Energies from Geant4\n(+ Fano noise)",
    figure = (size = (480,400), backgroundcolor = :transparent),
    axis = (xgridvisible = false, ygridvisible = false, 
        limits = (1585.5,1597,0.9,1050), #yscale = Makie.log10,
        xticks = 1586:2:1596,
        xlabel = "Energy / keV", ylabel = "Counts / 0.2 keV"
    ))
Makie.hist!(
    StatsBase.midpoints(first(h.edges)), 
    bins = first(h.edges),
    fillto = 1e-10,
    weights = replace(h.weights, 0 => 1e-10), 
    color = (:green, 0.3)
)

h_after = StatsBase.fit(Histogram, ustrip.(u"keV", Ectc), 0:0.2:3000)
Makie.stephist!(
    StatsBase.midpoints(first(h_after.edges)), 
    bins = first(h_after.edges), 
    label = "\nEnergies from\nsimulated waveforms\n(+ Fano noise)",
    weights = replace(h_after.weights, 0 => 1e-10), 
    color = :blue
)
Makie.hist!(
    StatsBase.midpoints(first(h_after.edges)), 
    bins = first(h_after.edges),
    fillto = 1e-10,
    weights = replace(h_after.weights, 0 => 1e-10), 
    color = (:blue, 0.3)
)

Makie.axislegend(ax, framevisible = false, position = :lt)
img = Makie.image!(fig.scene, rotr90(logo))
Makie.scale!(img,0.15,0.15)
Makie.translate!(img, (108,205))

# Makie.text!(2615, 0.3e4*2, text = L"$^{208}$Tl FEP", align = (:center, :bottom))
# Makie.text!(2615 - 511, 0.5e3*2, text = L"$^{208}$Tl SEP", align = (:center, :bottom))
save("Energy_shift.pdf", fig)
fig

In [ ]:
Δt = 1u"ns"
rt = @showprogress [ begin
E = last(wfi.signal)
RadiationDetectorDSP.Intersect()(wfi.signal, E * 0.90).x * Δt - RadiationDetectorDSP.Intersect()(wfi.signal, E * 0.005).x * Δt
end for wfi in wf.waveform];

In [ ]:
gflt = RadiationDetectorDSP.Gauss1DFilter(sigma = 1.0u"ns", length = 200.0u"ns")
flt_sg = RadiationDetectorDSP.SavitzkyGolayFilter(200.0u"ns", 3, 1)
flt = RadiationDetectorDSP.TrapezoidalChargeFilter(40u"ns", 100u"ns", 2000u"ns")
A = @showprogress [ begin
w = add_baseline_and_extend_tail(wfi, 10_000, 20_000)
w = SolidStateDetectors.RDWaveform(w.time, ustrip.(u"eV/V", w.signal .+ 1000*Unitful.q .* randn(length(w.signal))))
w = gflt(w)
w_sg = flt_sg(w)
w_flt = flt(w)
E = mean(w.signal[end-2000:end])
maximum(w_sg.signal), RadiationDetectorDSP.Intersect(2)(w_flt.signal, E * 0.90).x * u"ns" - RadiationDetectorDSP.Intersect(2)(w_flt.signal, E * 0.005).x * u"ns"
end for wfi in wf.waveform]

rt = getindex.(A,2)
AoE = ustrip.(u"eV^-1", getindex.(A,1) * 2.95 ./ Ectc)

AoE_median = median(AoE)

In [ ]:
jet_trunc = get(Makie.colorschemes[:jet], range(0, 0.65; length=256))
hh = StatsBase.fit(Histogram, (AoE ./ AoE_median, ustrip.(u"ns", rt)), (0.95:0.5e-3:1.05, 300:5:1650))
fig, ax, im = Makie.heatmap(hh.edges..., replace(hh.weights, 0.0=>NaN) ./ maximum(hh.weights), colorrange = (0.05,1), colormap = jet_trunc, colorscale = Makie.log10, nan_color = :transparent, 
    figure = (size = (480,400), backgroundcolor = :transparent), axis = (aspect = 1, xlabel = "A/E (a.u.)", ylabel = "0.5-90% Rise time (ns)", xgridvisible = false, ygridvisible = false))
Makie.Colorbar(fig[1,2], im, minorticksvisible = true, ticks = (exp10.(-2:0), ["0.01","0.1","1"]))
img = Makie.image!(fig.scene, Makie.rotr90(logo))
Makie.scale!(img,0.15,0.15)
Makie.translate!(img, (295,305))
FileIO.save("AoE_risetime_correlations.pdf", fig)
fig

In [ ]:
tcut = 1250u"ns"
fig = Makie.Figure(size = (600,400), backgroundcolor = :transparent)
ax = Makie.Axis(fig[1,1], xlabel = "0.5-90% Rise time (ns)", ylabel = "Counts", limits = (0,1650,0,900), xticks = 0:500:1500, yticks = 0:200:1000, xgridvisible = false, ygridvisible = false)
Makie.stephist!(ustrip.(u"ns", rt), bins = 50:10:1800, color = :black, linewidth = 2, label = "All events")
Makie.hist!(ustrip.(u"ns", rt[findall(10u"ns" .< rt .< tcut)]), bins = 50:10:1800, label = "Short rise time")
Makie.hist!(ustrip.(u"ns", rt[findall(rt .>= tcut)]), bins = 50:10:1800, label = "Long rise time")
Makie.axislegend(ax, "²⁰⁸Tl DEP", titlehalign = :left, position = :lt, framevisible = false)
ax2 = Makie.Axis(fig[1,2], xlabel = "A/E (a.u.)", limits = (0.98,1.03,0,900), xticks = 0.97:0.01:1.03, yticks = 0:200:1000, xgridvisible = false, ygridvisible = false)
Makie.stephist!(AoE ./ AoE_median, bins = 0.98:1e-3:1.03, color = :black, linewidth = 1.5)
Makie.hist!(AoE[findall(10u"ns" .< rt .< tcut)] ./ AoE_median, bins = 0.98:1e-3:1.03, color = (Makie.wong_colors()[1], 0.8))
Makie.hist!(AoE[findall(rt .>= tcut)] ./ AoE_median, bins = 0.98:1e-3:1.03, color = (Makie.wong_colors()[2], 0.8))
img = Makie.image!(fig.scene, Makie.rotr90(logo))
Makie.scale!(img,0.15,0.15)
Makie.translate!(img, (102,210))
img = Makie.image!(fig.scene, Makie.rotr90(logo))
Makie.scale!(img,0.15,0.15)
Makie.translate!(img, (475,310))
FileIO.save("AoE_risetime_histograms.pdf", fig)
fig